<a href="https://colab.research.google.com/github/faisu6339-glitch/LLMs/blob/main/Langchain(RUNNABLES).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Detailed Guide to LangChain Runnables and LCEL

In LangChain, a **Runnable** is a standard protocol that makes it easy to define, configure, and invoke custom chains. The **LangChain Expression Language (LCEL)** is built on this protocol to enable declarative chaining of various components.

### The Runnable Protocol

Most LangChain components implement the `Runnable` protocol, which provides a standard set of methods. This consistency ensures that components can be easily chained together using the pipe operator (`|`).

#### Standard Methods
Every Runnable implements these core synchronous and asynchronous methods:
* **`invoke` / `ainvoke`**: Call the runnable on a single input.
* **`batch` / `abatch`**: Call the runnable on a list of inputs (often optimized to run concurrently).
* **`stream` / `astream`**: Stream back chunks of the response as they are generated.

#### Core Types and Schema Methods
* **`input_schema`**: Dynamically tells you what structure of input the runnable expects (auto-generated from Pydantic).
* **`output_schema`**: Dynamically tells you what structure of output the runnable produces.

## Core Types of Runnables

Here are the most common and important Runnable wrappers and primitives used to build complex chains:

| Runnable Type | Purpose | Example Use Case |
| :--- | :--- | :--- |
| **`RunnableSequence`** | Chains multiple runnables sequentially where the output of one is the input to the next (created automatically via the `|` pipe operator). | `prompt | model | parser` |
| **`RunnableParallel`** | Executes multiple runnables in parallel, returning a dict of outputs. | Running multiple independent search queries simultaneously. |
| **`RunnablePassthrough`** | Passes input through unchanged, or adds extra keys to the input dictionary. | Formatting inputs before sending them to a prompt template. |
| **`RunnableLambda`** | Converts a standard Python function into a Runnable so it can be chained. | Custom text cleaning or mapping functions. |
| **`RunnableBranch`** | Implements conditional routing based on the input. | Routing to different sub-chains depending on classification. |
| **`RunnableBinding`** | Attaches runtime arguments or configurations (like stop sequences or tools) to an existing Runnable. | Binding a helper tool or setting temperature dynamically. |

## Code Example: Building and Using Runnables

Let's write a simple simulation demonstrating how these Runnables work in practice.

*Note: To run this code, make sure you have `langchain-core` installed (`pip install langchain-core`).*

In [6]:
# Install langchain-core if not already available
!pip install -q langchain-core

In [7]:
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

# 1. RunnableLambda: Turn custom Python functions into Runnables
def add_five(x: int) -> int:
    return x + 5

def multiply_by_three(x: int) -> int:
    return x * 3

runnable_add = RunnableLambda(add_five)
runnable_mult = RunnableLambda(multiply_by_three)

# 2. RunnableSequence: Chain them together using the pipe (|) operator
# Input -> add_five -> multiply_by_three -> Output
chain = runnable_add | runnable_mult

print("--- RunnableSequence Example ---")
print("Result of (10 + 5) * 3:", chain.invoke(10))

# 3. RunnableParallel: Execute operations in parallel and combine results into a dict
parallel_chain = RunnableParallel(
    original=RunnablePassthrough(),
    added_five=runnable_add,
    multiplied_three=runnable_mult
)

print("\n--- RunnableParallel Example ---")
display(parallel_chain.invoke(10))

# 4. Batch Processing
print("\n--- Batch Processing Example ---")
batch_results = chain.batch([2, 5, 8])
print("Results for batch [2, 5, 8]:", batch_results)

--- RunnableSequence Example ---
Result of (10 + 5) * 3: 45

--- RunnableParallel Example ---


{'original': 10, 'added_five': 15, 'multiplied_three': 30}


--- Batch Processing Example ---
Results for batch [2, 5, 8]: [21, 30, 39]


# 1.Simple Runnable Chain

prompt | model | parser

In [8]:


import os
from google.colab import userdata

api_key = userdata.get("GOOGLE_API_KEY")


from google import genai

client = genai.Client(
    api_key=api_key
)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say hello in one sentence."
)

print(response.text)

Hello there!


In [9]:
!pip install -U google-genai langchain-google-genai langchain-core


In [10]:

from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

#model Evaluation
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
    temperature=0.4
)

prompt = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple language."
)

parser = StrOutputParser()
chain = prompt | llm | parser

result = chain.invoke({
    "topic": "Machine Learning"
})

print(result)

Imagine you're trying to teach a computer to recognize a dog.

**The Old Way (Traditional Programming):**
You'd have to write down *every single rule* for what makes a dog a dog:
*   "If it has four legs AND fur AND barks AND has a tail..."
*   But what about a three-legged dog? Or a hairless dog? Or a dog that's not barking? You'd have to keep adding more and more specific rules, and it would be impossible to cover every single possibility.

**The Machine Learning Way:**
Instead of giving the computer a giant list of rules, you give it lots and lots of **examples**.

1.  **Data (Experience):** You show the computer thousands of pictures. Some are labeled "dog," and some are labeled "not a dog" (cats, birds, cars, etc.).
2.  **Training (Learning):** The computer then "studies" these pictures. It looks for patterns, features, and relationships that consistently appear in the "dog" pictures but not in the "not a dog" pictures. It essentially **learns** what a dog looks like by itself, wi

# Program:1. Explain a Topic

In [13]:
prompt = ChatPromptTemplate.from_template(
    """
    Explain the following topic in simple language.

    Topic: {topic}

    Include:
    1. Definition
    2. Simple explanation
    3. Example
    4. Real-world application
    """
)

# Create parser
parser = StrOutputParser()

In [14]:
#LCEL chain

chain=prompt | llm | parser

topic=input("Enter a topic: ")
result=chain.invoke({"topic":topic})
print("\n===== ANSWER =====")
print(result)

Enter a topic: Boobs

===== ANSWER =====
Let's break down the topic of "boobs" in simple language.

### Topic: Boobs

**1. Definition**
"Boobs" (more formally called **breasts**) are two rounded parts located on the chest of women and girls. They are made up of fatty tissue, connective tissue, and special glands and ducts designed to produce and deliver milk.

**2. Simple Explanation**
Think of boobs as natural milk factories that grow on a woman's chest. When a girl goes through puberty, hormones cause these areas to grow and develop. Inside, they have special glands and tubes (ducts) that can make and deliver milk once a woman has a baby. Besides their role in feeding infants, breasts are also a visible sign of maturity in women, and they come in a wide variety of shapes and sizes. Men also have breasts, but they are typically much smaller and usually don't develop the milk-producing capabilities.

**3. Example**
You can see examples of boobs (breasts) on almost any adult woman. For 

# 2 Question-Answer Chain

In [18]:

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
    temperature=0.4
)
prompt = ChatPromptTemplate.from_template(
    """
    You are a helpful AI teacher.

    Answer the following question clearly.

    Question:
    {question}

    Explain the answer with an example if necessary.
    """
)

parser = StrOutputParser()

chain = prompt | model | parser

question = input("Ask your question: ")

result = chain.invoke({
    "question": question
})

print("\n===== ANSWER =====")
print(result)

Ask your question: how to fix relationship

===== ANSWER =====
Fixing a relationship is one of the most challenging yet rewarding endeavors. It requires mutual effort, patience, and a genuine desire from both parties to make things better.

Here's a breakdown of how to approach fixing a relationship, explained with steps and examples:

---

### **Foundational Principles (Before You Start)**

1.  **Mutual Willingness:** Both individuals *must* want to fix the relationship. If only one person is trying, it's very difficult.
2.  **Self-Reflection:** Before pointing fingers, honestly consider your own role in the issues. What could *you* do differently?
3.  **Commitment & Patience:** Fixing deep-seated issues takes time and consistent effort. There will be ups and downs.

---

### **Steps to Fix a Relationship**

**1. Acknowledge and Address the Problem Directly**

*   **Identify Specific Issues:** Instead of vague complaints ("You always ignore me"), pinpoint specific behaviors or situati

# 3 Translation Chain
Goal
Translate an English sentence into Hindi.
English
   ↓
Prompt
   ↓
Gemini
   ↓
Hindi

In [19]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
    temperature=0.4
)
prompt = ChatPromptTemplate.from_template(
    """
    Translate the following English sentence into Hindi.

    English:
    {sentence}

    Return only the Hindi translation.
    """
)

parser = StrOutputParser()

chain = prompt | model | parser

sentence = input("Enter an English sentence: ")

result = chain.invoke({
    "sentence": sentence
})

print("\n===== HINDI TRANSLATION =====")
print(result)

Enter an English sentence: how are u

===== HINDI TRANSLATION =====
कैसे हो


#4.Code-Generation Chain
This is particularly useful for your Python/ML learning.
Goal
Give a requirement and ask Gemini to generate Python code.
Requirement
     ↓
Prompt
     ↓
Gemini
     ↓
Python Code

In [20]:
prompt = ChatPromptTemplate.from_template(
    """
    You are an expert Python programmer.

    Generate Python code for the following requirement:

    Requirement:
    {requirement}

    Requirements:
    - Use beginner-friendly Python.
    - Add comments.
    - Explain the code briefly after the code.
    """
)

parser = StrOutputParser()

chain = prompt | model | parser

requirement = input("Enter a requirement: ")

result = chain.invoke({
    "requirement": requirement
})

print("\n===== PYTHON CODE =====")
print(result)

Enter a requirement: Create a Python program to check whether a number is prime.

===== PYTHON CODE =====
```python
# --- Python Program to Check if a Number is Prime ---

# 1. Get input from the user
# The input() function gets text from the user.
# int() converts that text into a whole number (integer).
number = int(input("Enter a number: "))

# 2. Initialize a flag variable
# We assume the number is prime until we find evidence that it's not.
is_prime = True

# 3. Handle special cases for prime numbers
# Prime numbers are natural numbers greater than 1.

# Numbers less than or equal to 1 are not prime.
if number <= 1:
    is_prime = False
# The number 2 is the only even prime number.
elif number == 2:
    is_prime = True
# For numbers greater than 2, we need to check for divisors.
else:
    # 4. Loop to check for divisors
    # We only need to check for divisors from 2 up to half of the number.
    # If a number 'n' has a divisor 'd' greater than n/2,
    # then 'n/d' would be less 

# 5.Summarization Chain
This is a very important real-world application.
Goal
Take long text and generate a summary.
Long Text
    ↓
Prompt
    ↓
Gemini
    ↓
Summary

In [21]:
prompt = ChatPromptTemplate.from_template(
    """
    Summarize the following text.

    Text:
    {text}

    Requirements:
    - Keep the important information.
    - Remove unnecessary details.
    - Use simple language.
    - Give the summary in 5-7 sentences.
    """
)

parser = StrOutputParser()

chain = prompt | model | parser

text = input("Enter text to summarize: ")

result = chain.invoke({
    "text": text
})

print("\n===== SUMMARY =====")
print(result)

Enter text to summarize: Artificial Intelligence is a field of computer science...

===== SUMMARY =====
Artificial Intelligence (AI) is a field of computer science focused on creating machines that can think and act with human-like intelligence. Its primary goal is to enable computers to perform tasks that typically require human cognitive abilities. This includes capabilities such as learning from experience, understanding language, recognizing patterns, and solving complex problems. AI systems are designed to reason, make decisions, and adapt to new information. They can process vast amounts of data to identify trends and make predictions. AI is applied in many areas, including automation, healthcare, finance, and entertainment. Ultimately, AI aims to build intelligent agents that can perceive their environment and take actions to achieve specific goals.
